# 🗓️ Generador de Líneas de Tiempo — MIMP

> **¿Cómo usar esto?**
> 1. Haz clic en ▶ del primer bloque de código
> 2. Espera que aparezca el botón **SUBIR ARCHIVO**
> 3. Sube tu imagen, Word, PDF o Excel
> 4. ¡Descarga tu PowerPoint!

---

In [ ]:
# @title ▶ Haz clic aquí para iniciar
# @markdown Este bloque instala todo, muestra el botón para subir tu archivo
# @markdown y genera el PowerPoint automáticamente.

# ── 1. Instalación silenciosa ────────────────────────────────────────────────
import subprocess, sys, shutil, zipfile, urllib.request, os
from pathlib import Path
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets

display(HTML("<h3>⏳ Preparando el sistema... (30 segundos)</h3>"))

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "openpyxl", "python-pptx", "python-docx", "pdfplumber",
    "dateparser", "google-genai", "Pillow", "lxml",
    "ipywidgets"], check=True, capture_output=True)

# ── 2. Descargar código desde GitHub ─────────────────────────────────────────
ZIP_URL  = "https://github.com/KEVINmarce1996/timeline-generator-mimp/raw/main/timeline_generator_v2.zip"
proj_dir = Path("/content/timeline_generator")
if proj_dir.exists(): shutil.rmtree(proj_dir)
to_rm = [k for k in sys.modules if k.startswith(("core","utils"))]
for k in to_rm: del sys.modules[k]

zip_path = Path("/content/tg.zip")
urllib.request.urlretrieve(ZIP_URL, zip_path)
with zipfile.ZipFile(zip_path) as z:
    z.extractall("/content")
sys.path.insert(0, "/content/timeline_generator")

# ── 3. Cargar módulos ─────────────────────────────────────────────────────────
import importlib
import core.model_analyzer;  importlib.reload(core.model_analyzer)
import core.file_parser;      importlib.reload(core.file_parser)
import core.timeline_builder; importlib.reload(core.timeline_builder)
import core.pptx_generator;   importlib.reload(core.pptx_generator)
import utils.date_detector;   importlib.reload(utils.date_detector)

from core.model_analyzer   import ModelAnalyzer
from core.file_parser      import FileParser
from core.timeline_builder import TimelineBuilder, BuilderConfig
from core.pptx_generator   import PptxGenerator
from utils.date_detector   import DetectorConfig
from datetime import date

template = ModelAnalyzer(
    "/content/timeline_generator/assets/formato_excel_modelo.xlsx"
).extract()

# ── 4. Interfaz visual ────────────────────────────────────────────────────────
clear_output()

display(HTML("""
<div style="font-family:Arial; max-width:600px; margin:auto">
  <div style="background:#1F497D; color:white; padding:16px 20px;
              border-radius:8px 8px 0 0">
    <h2 style="margin:0">🗓️ Generador de Líneas de Tiempo</h2>
    <p style="margin:4px 0 0 0; opacity:0.85">MIMP — Proyectos de Inversión</p>
  </div>
  <div style="background:#f8f9fa; padding:16px 20px; border:1px solid #ddd;
              border-radius:0 0 8px 8px">
    <p style="color:#555; margin:0 0 12px 0">
      📎 <strong>Sube tu archivo</strong> con el cronograma del proyecto<br>
      <small style="color:#888">Formatos: imagen PNG/JPG, Word, PDF o Excel</small>
    </p>
  </div>
</div>
"""))

upload_btn = widgets.FileUpload(
    accept=".png,.jpg,.jpeg,.docx,.pdf,.xlsx,.xls",
    multiple=False,
    description="📂 Elegir archivo",
    button_style="primary",
    style={"button_width": "180px"},
    layout=widgets.Layout(width="220px", margin="8px auto")
)

status_out = widgets.Output()

def on_upload(change):
    if not upload_btn.value:
        return
    with status_out:
        clear_output()
        # Obtener el archivo subido
        file_info = list(upload_btn.value.values())[0]
        fname     = list(upload_btn.value.keys())[0]
        content   = file_info["content"]

        display(HTML(f"""
        <div style="font-family:Arial; max-width:600px; margin:8px auto;
                    background:#fff3cd; border:1px solid #ffc107;
                    border-radius:6px; padding:12px 16px">
          ⏳ Procesando <strong>{fname}</strong>...
        </div>
        """))

        try:
            # Guardar archivo
            upload_dir = Path("/content/mis_archivos")
            upload_dir.mkdir(exist_ok=True)
            dest = upload_dir / fname
            dest.write_bytes(bytes(content))

            # Procesar
            parser  = FileParser(DetectorConfig(granularity="annual"),
                                 use_vision=True)
            builder = TimelineBuilder(BuilderConfig(granularity="annual",
                                                    max_columns=14))
            parsed  = parser.parse(dest)
            layout  = builder.build(parsed)

            # Generar PPTX
            out_dir = Path("/content/output"); out_dir.mkdir(exist_ok=True)
            pptx_path = out_dir / "timeline.pptx"
            gen = PptxGenerator(template)
            gen.add_layout(layout)
            gen.save(pptx_path)

            # Resumen de hitos
            rows = ""
            for s in layout.sections:
                for col in s.columns:
                    for e in col.events:
                        d = e.detected_date.date_start.strftime("%d/%m/%Y") \
                            if e.detected_date and e.detected_date.date_start else "-"
                        color = {"done":"#6c757d","active":"#1F497D",
                                 "pending":"#856404"}.get(e.status,"#333")
                        bg    = {"done":"#f8f9fa","active":"#cfe2ff",
                                 "pending":"#fff3cd"}.get(e.status,"#fff")
                        estado= {"done":"Ejecutado","active":"En curso",
                                 "pending":"Pendiente"}.get(e.status,"")
                        rows += (f'<tr style="background:{bg};">'
                                 f'<td style="padding:4px 8px;color:{color};'
                                 f'font-weight:bold">{estado}</td>'
                                 f'<td style="padding:4px 8px">{d}</td>'
                                 f'<td style="padding:4px 8px">{e.label[:50]}</td>'
                                 f'</tr>')

            display(HTML(f"""
            <div style="font-family:Arial; max-width:600px; margin:8px auto">
              <div style="background:#d1e7dd; border:1px solid #a3cfbb;
                          border-radius:6px; padding:12px 16px; margin-bottom:12px">
                ✅ <strong>¡Línea de tiempo generada!</strong><br>
                <small>Proyecto: {layout.project_title}</small>
              </div>
              <table style="width:100%;border-collapse:collapse;
                            font-size:12px;border:1px solid #dee2e6;
                            border-radius:6px;overflow:hidden">
                <thead>
                  <tr style="background:#1F497D;color:white">
                    <th style="padding:6px 8px;text-align:left">Estado</th>
                    <th style="padding:6px 8px;text-align:left">Fecha</th>
                    <th style="padding:6px 8px;text-align:left">Etapa</th>
                  </tr>
                </thead>
                <tbody>{rows}</tbody>
              </table>
            </div>
            """))

            # Descargar automáticamente
            from google.colab import files as cf
            cf.download(str(pptx_path))

            display(HTML("""
            <div style="font-family:Arial;max-width:600px;margin:8px auto;
                        background:#cfe2ff;border:1px solid #9ec5fe;
                        border-radius:6px;padding:12px 16px;text-align:center">
              📥 <strong>El PowerPoint se está descargando</strong><br>
              <small>Revisa tu carpeta de Descargas</small>
            </div>
            """))

        except Exception as ex:
            import traceback
            tb = traceback.format_exc()
            display(HTML(f"""
            <div style="font-family:Arial;max-width:600px;margin:8px auto;
                        background:#f8d7da;border:1px solid #f5c2c7;
                        border-radius:6px;padding:12px 16px">
              ❌ <strong>Error procesando el archivo</strong><br>
              <small><pre style="white-space:pre-wrap">{tb[:500]}</pre></small>
            </div>
            """))

upload_btn.observe(on_upload, names="value")

display(widgets.VBox(
    [upload_btn, status_out],
    layout=widgets.Layout(align_items="center", margin="0 auto")
))
